# GAN Example 2 (PyTorch) — Conditional GAN

PyTorch version of `conditional_gan.ipynb`. By feeding the **class label** into both the
generator and discriminator, we can ask the model to draw a *specific* digit.

- **Generator** input = noise **+** one-hot label.
- **Discriminator** input = image **+** one-hot label (judges "real image *of this
  class*?").

A compact MLP GAN keeps training fast on CPU.

> Needs `torch` + `torchvision`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

In [ ]:
NUM_CLASSES, NOISE_DIM, BATCH_SIZE = 10, 100, 256
# ToTensor -> [0,1]; Normalize -> [-1,1]; then we flatten in the loop
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print('Train:', len(train_dataset))

## Models (label-conditioned MLPs)

Both concatenate a one-hot label onto their input.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(NOISE_DIM + NUM_CLASSES, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 784), nn.Tanh(),
        )
    def forward(self, z, onehot):
        return self.net(torch.cat([z, onehot], dim=1))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784 + NUM_CLASSES, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),
        )
    def forward(self, img, onehot):
        return self.net(torch.cat([img, onehot], dim=1))

G = Generator().to(device)
D = Discriminator().to(device)
criterion = nn.BCEWithLogitsLoss()
g_opt = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
def show_each_digit(epoch=None):
    G.eval()
    with torch.no_grad():
        onehot = F.one_hot(torch.arange(10), NUM_CLASSES).float().to(device)
        imgs = G(torch.randn(10, NOISE_DIM, device=device), onehot).cpu().numpy().reshape(-1, 28, 28)
    G.train()
    plt.figure(figsize=(12, 1.6))
    for i in range(10):
        plt.subplot(1, 10, i + 1)
        plt.imshow(imgs[i] * 0.5 + 0.5, cmap='gray'); plt.axis('off'); plt.title(str(i))
    if epoch is not None:
        plt.suptitle(f'Epoch {epoch}')
    plt.show()

## Train

In [ ]:
EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    for imgs, labels in train_loader:
        bsz = imgs.size(0)
        imgs = imgs.view(bsz, -1).to(device)
        onehot = F.one_hot(labels, NUM_CLASSES).float().to(device)
        ones, zeros = torch.ones(bsz, 1, device=device), torch.zeros(bsz, 1, device=device)

        # Discriminator
        z = torch.randn(bsz, NOISE_DIM, device=device)
        fake = G(z, onehot)
        d_loss = criterion(D(imgs, onehot), ones) + criterion(D(fake.detach(), onehot), zeros)
        d_opt.zero_grad(); d_loss.backward(); d_opt.step()

        # Generator
        g_loss = criterion(D(fake, onehot), ones)
        g_opt.zero_grad(); g_loss.backward(); g_opt.step()

    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={g_loss.item():.3f}  disc_loss={d_loss.item():.3f}')
    if epoch % 5 == 0 or epoch == 1:
        show_each_digit(epoch)

## Generate on demand

In [ ]:
WANT = 7  # <-- change me
G.eval()
with torch.no_grad():
    onehot = F.one_hot(torch.full((10,), WANT), NUM_CLASSES).float().to(device)
    imgs = G(torch.randn(10, NOISE_DIM, device=device), onehot).cpu().numpy().reshape(-1, 28, 28)
plt.figure(figsize=(12, 1.6))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(imgs[i] * 0.5 + 0.5, cmap='gray'); plt.axis('off')
plt.suptitle(f'Ten generated "{WANT}"s')
plt.show()

### Things to try

- Use an `nn.Embedding` for the label instead of one-hot.
- Make it convolutional for sharper digits.
- Try label smoothing (`0.9` for real) to stabilise training.